In [1]:
import json

image_root = "/mnt/petrelfs/share_data/liqingyun/data/MME-RealWorld/"
annotation_path = "/mnt/petrelfs/share_data/liqingyun/data/MME-RealWorld/MME_RealWorld_remote_sensing.json"

with open(annotation_path, "r") as f:
    data = json.load(f)

data[0]

{'Question_id': 'perception/remote_sensing/color/0001',
 'Image': 'remote_sensing/03553_Toronto.png',
 'Text': 'What color is the roof of the square building in the lower right area of the picture?',
 'Question Type': 'Multiple Choice',
 'Answer choices': ['(A) Yellow',
  '(B) Blue',
  '(C) Gray',
  '(D) White',
  '(E) The image does not feature the color.'],
 'Ground truth': 'D',
 'Category': 'color',
 'Subtask': 'Remote Sensing',
 'Task': 'Perception',
 'size': [11500, 7500]}

## 查看prompt里是否有显示的方位限定？

感觉可以直接走一轮简单的crop，直接到限定的方位上

In [2]:
import re
import random

def check_in(txt: str, check_list: list) -> bool:
    for c in check_list:
        if re.search(c, txt):
            return True

rest_data = []
for sample in data:
    prompt = sample["Text"]
    if check_in(prompt.lower().strip(), [
        r'(far )?(top|bottom|left|right|middle|center|upper|lower)( area| side| edge| row| part)* (of|in|above|below) (the|this)( )+(picture|pisture)',
        r'(top|bottom|left|right|middle|center|upper|lower)( |-)(right|left|middle)', r'middle( |-)(right|left|bottom|top)', 
        'center area', 
        r'the leftmost side of the bottom edge of (the|this) (picture|image)', 
        r'the left of the center of (the|this) (picture|image)',
        r'the middle of the top (picture|image)',
        r'the leftmost row of the (picture|image)',
        r'the middle area of the edge in the (picture|image)',
    ]):
        continue
    rest_data.append(sample)

print(len(data))
print(len(rest_data))
example = random.choice(rest_data)
print(example['Text'])
print(example['Answer choices'])
print(example['Ground truth'])

3738
1709
Where is the triangular area surrounded by buildings in the picture?
['(A) In the upper left area of the picture', '(B) In the upper right area of the picture', '(C) In the lower left area of the picture', '(D) In the lower right area of the picture', "(E) This image doesn't feature the position."]
B


## 查看position的选项都是啥

In [88]:
position_data = [s for s in data if s['Category'] == 'position']
print(len(position_data))

options = set()
for sample in position_data:
    response = sample['Answer choices'][0]
    choice = [e for e in re.split(r'(\([ABCD]\) )', response) if e.strip()]
    options.update(choice)

print(len(options))
for i, option in enumerate(options):
    print(f"{i+1}: {option}")

1257
319
1: In the top left area of this picture
2: In the lower left area of the picture
3: In the bottom left of the picture
4: In the right area of the picture
5: Above the upper water surface on the left side of the picture.
6: Near the water in the upper left area of this picture
7: In the bottom right corner of the right half of this picture
8: To the right of the picture, near the white building by the water's edge
9: On the top right side of the picture.
10: On the side of the horizontal road in the middle of the picture, between the green belt and the row of buildings.
11: Bottom area of the picture.
12: In the middle of the right edge of the picture.
13: In the upper left area in the center of the image
14: In the right area of the picture.
15: In the upper right of the picture.
16: In the middle of the left edge of the picture.
17: At the bottom in the middle of this picture
18: In the central area of the picture.
19: above the picture
20: Left edge of the image
21: In the b

## 查看color的选项都是啥

In [104]:
color_data = [s for s in data if s['Category'] == 'color']
print(len(color_data))

options = set()
for sample in color_data:
    response = sample['Answer choices'][0].strip().rstrip(".")
    choice = [e.strip().lower() for e in re.split(r'\([ABCD]\) ', response) if e.strip()]
    options.update(choice)

options = sorted(list(options), key=lambda x: (len(x), x))
print(len(options))
for i, option in enumerate(options):
    print(f"{i+1}: {option}")

1255
107
1: red
2: yes
3: blue
4: cyan
5: gray
6: grey
7: pink
8: black
9: brown
10: green
11: white
12: coffee
13: golden
14: orange
15: purple
16: silver
17: sliver
18: yellow
19: grizzly
20: dark red
21: brick-red
22: grey blue
23: light red
24: dark green
25: light blue
26: light pink
27: light brown
28: silver grey
29: blue and red
30: brick-yellow
31: brownish red
32: light yellow
33: red and blue
34: red and gray
35: red and grey
36: silver white
37: blue and grey
38: bright yellow
39: gray and blue
40: green and red
41: red and black
42: red and green
43: red and white
44: white and red
45: black and blue
46: black and grey
47: black and pink
48: blue and green
49: blue and white
50: brown and blue
51: brown and grey
52: green and blue
53: green and gray
54: green and grey
55: grey and black
56: grey and brown
57: grey and green
58: grey and white
59: pink and white
60: white and blue
61: white and grey
62: yellow and red
63: black and white
64: blue and yellow
65: brown and gr